# Continuous RNA reactivity and degradation model

Train a compact, structure-aware convolution + transformer encoder on **OpenVaccine**.
Each nucleotide gets a continuous embedding and three predictions: `reactivity`,
`deg_Mg_pH10`, and `deg_Mg_50C`. This is a supervised starting model inspired by
[RNAdegformer](https://github.com/Shujun-He/RNAdegformer), not a reproduction of its architecture or results.

The notebook uses the existing sequence-grouped `split.csv` from `leakage_train_test.ipynb`.
Training and model selection never read test labels; the final evaluation section loads them once the best checkpoint is selected. The full sequence is input, but only finite labels inside
`seq_scored` contribute to loss and metrics. Both partitions use `SN_filter == 1` by default.

Run top to bottom. The default is at most 50 epochs with early stopping; set
`CFG["epochs"] = 2` for a quick run. Outputs go to a fresh run directory under
`Project/outputs/continuous_rna/`. No model has been pretrained on Ribonanza here.

**BPP inputs:** load `bpps/<rna_id>.npy` for nucleotide pairing summaries and full pairwise attention bias. Restart the kernel and run all cells to train this updated architecture. Previous results remain in `continuous_rna_before_bpp.ipynb` and existing run directories.


In [ ]:
# Run once in your notebook kernel if these dependencies are missing:
# %pip install torch numpy pandas matplotlib

import json
import math
import random
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

CFG = dict(
    seed=42, batch_size=32, epochs=50, patience=10,
    learning_rate=3e-4, weight_decay=1e-2, grad_clip=1.0,
    d_model=128, n_heads=4, n_layers=4, ff_multiplier=4,
    dropout=0.1, conv_kernel=5, high_quality_only=True,
    use_bpp=True,  # False gives the original model without BPP inputs
    device="auto",  # or "cpu", "cuda", "mps"
)
TARGETS = ["reactivity", "deg_Mg_pH10", "deg_Mg_50C"]
BPP_FEATURE_NAMES = ["paired_probability", "unpaired_probability",
                     "max_pair_probability", "normalized_pairing_entropy",
                     "expected_normalized_pair_distance"]

random.seed(CFG["seed"])
np.random.seed(CFG["seed"])
torch.manual_seed(CFG["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG["seed"])
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
# Seeds improve repeatability; different devices can still give different results.
auto_device = "cuda" if torch.cuda.is_available() else (
    "mps" if torch.backends.mps.is_available() else "cpu"
)
device = torch.device(auto_device if CFG["device"] == "auto" else CFG["device"])
print("PyTorch:", torch.__version__, "| device:", device)


## Load the existing split

Resolve paths from either the workspace root or a notebook directory. The original
`train.json` is the source of labels; `split.csv` supplies fixed memberships.
Validate IDs and clusters before quality filtering, so filtering cannot change assignments.


In [ ]:
roots = [Path.cwd(), *Path.cwd().parents]
candidates = [p / suffix for p in roots for suffix in
              ("data/stanford-covid-vaccine", "Project/data/stanford-covid-vaccine")]
DATA = next((p.resolve() for p in candidates if (p / "train.json").exists()), None)
if DATA is None:
    raise FileNotFoundError("Cannot find Project/data/stanford-covid-vaccine/train.json")
if not (DATA / "split.csv").exists():
    raise FileNotFoundError("Run the split section of leakage_train_test.ipynb first.")

raw = pd.read_json(DATA / "train.json", lines=True)
assignment = pd.read_csv(DATA / "split.csv", dtype={"id": str})
assert raw.id.is_unique and assignment.id.is_unique, "Duplicate sequence IDs"
assert set(assignment.id) == set(raw.id), "Split IDs do not match train.json"
assert set(assignment.split) == {"train", "val"}
assert assignment.groupby("cluster")["split"].nunique().max() == 1, "Cluster crosses split"
frame = raw.merge(assignment, on="id", validate="one_to_one")
if CFG["high_quality_only"]:
    frame = frame.loc[frame.SN_filter.eq(1)].copy()
train_df = frame.loc[frame.split.eq("train")].reset_index(drop=True)
val_df = frame.loc[frame.split.eq("val")].reset_index(drop=True)
assert len(train_df) > 0 and len(val_df) > 0
assert set(train_df.id).isdisjoint(set(val_df.id))
assert set(train_df.cluster).isdisjoint(set(val_df.cluster))
assert set(train_df.sequence).isdisjoint(set(val_df.sequence))

# Independently verify the saved split's 80% variable-region identity rule.
# These OpenVaccine training constructs have a 5-nt prefix and 21-nt suffix.
assert frame.seq_length.eq(107).all(), "Revisit similarity check for a different dataset"
train_variable = np.array([list(s[5:-21]) for s in train_df.sequence])
val_variable = np.array([list(s[5:-21]) for s in val_df.sequence])
max_cross_matches = max(
    int((val_variable[start:start+32, None] == train_variable[None]).sum(-1).max())
    for start in range(0, len(val_variable), 32)
)
assert max_cross_matches < 65, "Saved split violates its >=65/81 matching rule"
print(f"Train: {len(train_df)} RNAs | validation: {len(val_df)} RNAs")
print(f"Largest cross-split variable-region identity: {max_cross_matches / 81:.3f}")

PROJECT = DATA.parent.parent
RUN_DIR = PROJECT / "outputs" / "continuous_rna" / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
RUN_DIR.mkdir(parents=True, exist_ok=False)
(RUN_DIR / "config.json").write_text(json.dumps(CFG, indent=2))
frame[["id", "split", "cluster"]].to_csv(RUN_DIR / "used_split.csv", index=False)
print("Run directory:", RUN_DIR)


## Tokenize RNA and load base-pair probabilities

Zero is reserved for padding. Dot-bracket pairs remain a separate binary connection
matrix. Each RNA's `bpps/<id>.npy` supplies the probability matrix P, with P[i,j]
the predicted probability of bases i and j pairing. These inputs come from the
provided structural predictions, not the experimental target labels.

The five per-nucleotide features are:
- Paired probability: sum of P[i,j] over j.
- Unpaired probability: 1 minus paired probability.
- Maximum pair probability: maximum over potential partners j.
- Pairing entropy: entropy over all partner probabilities plus the unpaired state,
  divided by log(L+1), where L is the unpadded RNA length.
- Expected normalized pair distance: sum of P[i,j] * |i-j|/(L-1).
  This is unconditional: the unpaired state contributes zero.

The loader checks dimensions, finite values, probability bounds, symmetry, zero diagonal
and row sums. Missing or invalid BPP files raise a clear error instead of silently
substituting zeros. Summaries are computed before padding. `use_bpp=False` skips BPP loading.


In [ ]:
VOCABS = {
    "sequence": {c: i + 1 for i, c in enumerate("ACGU")},
    "structure": {c: i + 1 for i, c in enumerate(".()")},
    "predicted_loop_type": {c: i + 1 for i, c in enumerate("SHIBMXE")},
}

def pair_matrix(structure):
    pairs = torch.zeros(len(structure), len(structure), dtype=torch.float32)
    stack = []
    for i, char in enumerate(structure):
        if char == "(":
            stack.append(i)
        elif char == ")":
            if not stack:
                raise ValueError("Unbalanced dot-bracket structure")
            j = stack.pop()
            pairs[i, j] = pairs[j, i] = 1.0
        elif char != ".":
            raise ValueError(f"Unsupported structure symbol: {char}")
    if stack:
        raise ValueError("Unbalanced dot-bracket structure")
    return pairs

def load_bpp_features(bpp_dir, rna_id, length):
    path = Path(bpp_dir) / f"{rna_id}.npy"
    if not path.exists():
        raise FileNotFoundError(f"Missing BPP input for {rna_id}: {path}")
    probabilities = np.asarray(np.load(path, allow_pickle=False), dtype=np.float32)
    if probabilities.shape != (length, length):
        raise ValueError(f"BPP shape mismatch for {rna_id}: {probabilities.shape}")
    if not np.isfinite(probabilities).all():
        raise ValueError(f"Non-finite BPP input for {rna_id}")
    tolerance = 1e-5
    if probabilities.min() < -tolerance or probabilities.max() > 1 + tolerance:
        raise ValueError(f"BPP probabilities outside [0, 1] for {rna_id}")
    if not np.allclose(probabilities, probabilities.T, atol=tolerance, rtol=0):
        raise ValueError(f"Asymmetric BPP matrix for {rna_id}")
    if np.abs(np.diag(probabilities)).max() > tolerance:
        raise ValueError(f"Nonzero BPP diagonal for {rna_id}")
    probabilities = np.clip(probabilities, 0, 1)
    probabilities = (probabilities + probabilities.T) * 0.5
    np.fill_diagonal(probabilities, 0)
    paired = probabilities.sum(axis=1)
    if paired.max() > 1 + tolerance:
        raise ValueError(f"BPP row sum exceeds 1 for {rna_id}")
    # Only correct tiny float rounding excursions; preserve matrix symmetry.
    probabilities /= max(1.0, float(paired.max()))
    paired = probabilities.sum(axis=1)
    unpaired = np.clip(1.0 - paired, 0.0, 1.0)
    entropy = -(probabilities * np.log(np.maximum(probabilities, 1e-12))).sum(axis=1)
    entropy -= unpaired * np.log(np.maximum(unpaired, 1e-12))
    entropy /= np.log(length + 1)
    positions = np.arange(length, dtype=np.float32)
    distance = np.abs(positions[:, None] - positions[None, :]) / max(length - 1, 1)
    expected_distance = (probabilities * distance).sum(axis=1)
    features = np.stack([paired, unpaired, probabilities.max(axis=1),
                         entropy, expected_distance], axis=-1).astype(np.float32)
    return torch.from_numpy(probabilities.copy()), torch.from_numpy(features)

class RNADataset(Dataset):
    def __init__(self, dataframe, bpp_dir, use_bpp=True):
        self.items = []
        for row in dataframe.to_dict("records"):
            length, scored = int(row["seq_length"]), int(row["seq_scored"])
            assert 0 < scored <= length
            assert all(len(row[name]) == length for name in VOCABS)
            tokens = torch.tensor([
                [vocab[c] for c in row[name]] for name, vocab in VOCABS.items()
            ], dtype=torch.long).T
            labels = torch.zeros(length, len(TARGETS), dtype=torch.float32)
            observed = torch.zeros_like(labels, dtype=torch.bool)
            for k, target in enumerate(TARGETS):
                values = np.asarray(row[target], dtype=np.float32)
                assert len(values) in (scored, length), f"Unexpected target length for {row['id']}"
                # Test labels include a null-padded tail; training labels stop at seq_scored.
                values = values[:scored]
                finite = np.isfinite(values)
                labels[:scored, k] = torch.from_numpy(np.where(finite, values, 0.0))
                observed[:scored, k] = torch.from_numpy(finite)
            if use_bpp:
                bpp, bpp_features = load_bpp_features(bpp_dir, row["id"], length)
            else:
                bpp = torch.zeros(length, length)
                bpp_features = torch.zeros(length, len(BPP_FEATURE_NAMES))
            self.items.append(dict(
                id=row["id"], tokens=tokens, pairs=pair_matrix(row["structure"]),
                labels=labels, observed=observed, bpp=bpp, bpp_features=bpp_features,
            ))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        return self.items[index]

def collate_rna(items):
    batch_size = len(items)
    length = max(len(item["tokens"]) for item in items)
    batch = dict(
        ids=[item["id"] for item in items],
        tokens=torch.zeros(batch_size, length, 3, dtype=torch.long),
        pairs=torch.zeros(batch_size, length, length),
        bpp=torch.zeros(batch_size, length, length),
        bpp_features=torch.zeros(batch_size, length, len(BPP_FEATURE_NAMES)),
        valid=torch.zeros(batch_size, length, dtype=torch.bool),
        labels=torch.zeros(batch_size, length, len(TARGETS)),
        observed=torch.zeros(batch_size, length, len(TARGETS), dtype=torch.bool),
    )
    for i, item in enumerate(items):
        n = len(item["tokens"])
        batch["tokens"][i, :n] = item["tokens"]
        batch["pairs"][i, :n, :n] = item["pairs"]
        batch["bpp"][i, :n, :n] = item["bpp"]
        batch["bpp_features"][i, :n] = item["bpp_features"]
        batch["valid"][i, :n] = True
        batch["labels"][i, :n] = item["labels"]
        batch["observed"][i, :n] = item["observed"]
    return batch

BPP_DIR = DATA / "bpps"
train_ds = RNADataset(train_df, BPP_DIR, use_bpp=CFG["use_bpp"])
val_ds = RNADataset(val_df, BPP_DIR, use_bpp=CFG["use_bpp"])
generator = torch.Generator().manual_seed(CFG["seed"])
train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True,
                          generator=generator, collate_fn=collate_rna, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=CFG["batch_size"], shuffle=False,
                        collate_fn=collate_rna, num_workers=0)
print("First training sample:", train_ds[0]["tokens"].shape,
      "| labeled values:", int(train_ds[0]["observed"].sum()))

if CFG["use_bpp"]:
    print("BPP matrix:", train_ds[0]["bpp"].shape, "| nucleotide features:", BPP_FEATURE_NAMES)


## Continuous encoder with BPP inputs

The encoder sums sequence/structure/loop embeddings with a learned linear projection
of the five BPP features, then adds sinusoidal positions. A residual convolution
and transformer blocks produce one continuous vector per nucleotide.

Each attention block has separate learned per-head coefficients for the binary
dot-bracket pair matrix and the full BPP matrix. Thus alternative partners affect
attention directly, rather than being reduced to a single predicted fold. All padded
keys are masked, and padded node features are zeroed before the convolution.

`model.encode(tokens, pairs, valid, bpp, bpp_features)` returns continuous embeddings.
The identity bottleneck remains available for the later FSQ experiment. This is a
compact BPP-aware model, not the complete RNAdegformer pairwise-convolution architecture.


In [ ]:
class PairAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout, use_bpp=False):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.output = nn.Linear(d_model, d_model)
        self.pair_bias = nn.Parameter(torch.ones(n_heads))
        self.bpp_bias = nn.Parameter(torch.ones(n_heads)) if use_bpp else None
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, pairs, valid, bpp=None):
        b, length, width = x.shape
        qkv = self.qkv(x).reshape(b, length, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4).unbind(0)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        scores = scores + self.pair_bias[None, :, None, None] * pairs[:, None]
        if self.bpp_bias is not None:
            scores = scores + self.bpp_bias[None, :, None, None] * bpp[:, None]
        scores = scores.masked_fill(~valid[:, None, None, :], float("-inf"))
        weights = self.dropout(scores.softmax(dim=-1))
        out = (weights @ v).transpose(1, 2).reshape(b, length, width)
        return self.output(out).masked_fill(~valid[..., None], 0.0)

class EncoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d = cfg["d_model"]
        self.norm1, self.norm2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.attention = PairAttention(d, cfg["n_heads"], cfg["dropout"], cfg.get("use_bpp", False))
        self.ff = nn.Sequential(
            nn.Linear(d, cfg["ff_multiplier"] * d), nn.GELU(),
            nn.Dropout(cfg["dropout"]), nn.Linear(cfg["ff_multiplier"] * d, d),
        )
        self.dropout = nn.Dropout(cfg["dropout"])

    def forward(self, x, pairs, valid, bpp=None):
        x = x + self.dropout(self.attention(self.norm1(x), pairs, valid, bpp))
        x = x + self.dropout(self.ff(self.norm2(x)))
        return x.masked_fill(~valid[..., None], 0.0)

class ContinuousRNA(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d, kernel = cfg["d_model"], cfg["conv_kernel"]
        assert d % 2 == 0 and kernel % 2 == 1
        self.embeddings = nn.ModuleList([
            nn.Embedding(len(vocab) + 1, d, padding_idx=0) for vocab in VOCABS.values()
        ])
        self.use_bpp = cfg.get("use_bpp", False)
        self.bpp_projection = nn.Linear(len(BPP_FEATURE_NAMES), d, bias=False) if self.use_bpp else None
        self.input_dropout = nn.Dropout(cfg["dropout"])
        self.conv_norm = nn.LayerNorm(d)
        self.conv = nn.Conv1d(d, d, kernel, padding=kernel // 2)
        self.activation = nn.GELU()
        self.blocks = nn.ModuleList([EncoderBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm = nn.LayerNorm(d)
        self.bottleneck = nn.Identity()
        self.head = nn.Sequential(nn.Linear(d, d // 2), nn.GELU(),
                                  nn.Dropout(cfg["dropout"]), nn.Linear(d // 2, len(TARGETS)))

    def encode(self, tokens, pairs, valid, bpp=None, bpp_features=None):
        x = sum(embedding(tokens[..., i]) for i, embedding in enumerate(self.embeddings))
        if self.use_bpp:
            if bpp is None or bpp_features is None:
                raise ValueError("This checkpoint requires BPP matrices and nucleotide features")
            x = x + self.bpp_projection(bpp_features)
        length, width = x.shape[1:]
        position = torch.arange(length, device=x.device, dtype=x.dtype)[:, None]
        frequency = torch.exp(torch.arange(0, width, 2, device=x.device, dtype=x.dtype)
                              * (-math.log(10000.0) / width))
        pe = torch.zeros(length, width, device=x.device, dtype=x.dtype)
        pe[:, 0::2], pe[:, 1::2] = torch.sin(position * frequency), torch.cos(position * frequency)
        x = self.input_dropout(x + pe).masked_fill(~valid[..., None], 0.0)
        conv_input = self.conv_norm(x).masked_fill(~valid[..., None], 0.0)
        local = self.conv(conv_input.transpose(1, 2)).transpose(1, 2)
        x = (x + self.input_dropout(self.activation(local))).masked_fill(~valid[..., None], 0.0)
        for block in self.blocks:
            x = block(x, pairs, valid, bpp)
        return self.final_norm(x).masked_fill(~valid[..., None], 0.0)

    def forward(self, tokens, pairs, valid, bpp=None, bpp_features=None):
        return self.head(self.bottleneck(self.encode(tokens, pairs, valid, bpp, bpp_features)))

model = ContinuousRNA(CFG).to(device)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters()):,}")


## Loss and epoch loops

Training minimizes the mean of the observed per-target MSEs. Validation MCRMSE is
computed **once from accumulated squared errors and counts across the entire set**,
not by averaging batch RMSEs. The two-target degradation MCRMSE is also reported.
All reported scores are unweighted; experimental errors are not input features.


In [ ]:
def move_batch(batch, device):
    return {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}

def masked_mse(predictions, labels, observed):
    squared = (predictions - labels).square().masked_fill(~observed, 0.0)
    counts = observed.sum(dim=(0, 1))
    present = counts > 0
    if not present.any():
        raise ValueError("Batch has no observed targets")
    return (squared.sum(dim=(0, 1))[present] / counts[present]).mean()

def run_epoch(model, loader, device, optimizer=None, grad_clip=1.0):
    training = optimizer is not None
    model.train(training)
    # CPU float64 accumulation also works when the accelerator lacks float64 (MPS).
    squared_sum = torch.zeros(len(TARGETS), dtype=torch.float64)
    counts = torch.zeros(len(TARGETS), dtype=torch.float64)
    with torch.set_grad_enabled(training):
        for cpu_batch in loader:
            batch = move_batch(cpu_batch, device)
            if training:
                optimizer.zero_grad(set_to_none=True)
            pred = model(batch["tokens"], batch["pairs"], batch["valid"],
                         batch["bpp"], batch["bpp_features"])
            if not torch.isfinite(pred[batch["valid"]]).all():
                raise FloatingPointError("Non-finite model predictions")
            loss = masked_mse(pred, batch["labels"], batch["observed"])
            if training:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), grad_clip, error_if_nonfinite=True)
                optimizer.step()
            errors = (pred.detach() - batch["labels"]).square().masked_fill(~batch["observed"], 0.0)
            squared_sum += errors.cpu().double().sum(dim=(0, 1))
            counts += batch["observed"].cpu().double().sum(dim=(0, 1))
    if (counts == 0).any():
        raise ValueError("A target has no observed values in this partition")
    per_target_mse = squared_sum / counts
    rmse = per_target_mse.sqrt()
    metrics = {f"rmse_{target}": float(rmse[k]) for k, target in enumerate(TARGETS)}
    metrics.update(mse=float(per_target_mse.mean()), mcrmse=float(rmse.mean()),
                   degradation_mcrmse=float(rmse[1:].mean()))
    return metrics

# A train-only mean predictor gives a cheap reference on exactly this split.
train_labels = torch.cat([item["labels"] for item in train_ds.items])
train_observed = torch.cat([item["observed"] for item in train_ds.items])
target_means = (train_labels * train_observed).sum(0) / train_observed.sum(0)
class MeanPredictor(nn.Module):
    def __init__(self, means):
        super().__init__()
        self.register_buffer("means", means)
    def forward(self, tokens, pairs, valid, bpp=None, bpp_features=None):
        return self.means.expand(*tokens.shape[:2], -1)

mean_baseline = run_epoch(MeanPredictor(target_means).to(device), val_loader, device)
print("Validation mean-predictor baseline:", mean_baseline)


## Train, validate, and save the best model

Selection uses three-target validation MCRMSE. Learning rate is reduced on a plateau;
early stopping counts epochs without a meaningful improvement. The saved training MSE
uses predictions collected during the epoch as the model changes; validation uses the
fixed end-of-epoch model. Rerunning this cell starts training from scratch.


In [ ]:
def fit(model, train_loader, val_loader, cfg, device, run_dir):
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["learning_rate"],
                                 weight_decay=cfg["weight_decay"])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=3,
    )
    best_score, stale = float("inf"), 0
    history = []
    best_path = run_dir / "best_model.pt"
    for epoch in range(1, cfg["epochs"] + 1):
        started = time.perf_counter()
        lr = optimizer.param_groups[0]["lr"]
        train_metrics = run_epoch(model, train_loader, device, optimizer, cfg["grad_clip"])
        val_metrics = run_epoch(model, val_loader, device)
        score = val_metrics["mcrmse"]
        scheduler.step(score)
        row = dict(epoch=epoch, learning_rate=lr, seconds=time.perf_counter() - started)
        row.update({f"train_{k}": v for k, v in train_metrics.items()})
        row.update({f"val_{k}": v for k, v in val_metrics.items()})
        history.append(row)
        pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)
        improved = score < best_score - 1e-6
        if improved:
            best_score, stale = score, 0
            torch.save({
                "model_state": {k: v.detach().cpu() for k, v in model.state_dict().items()},
                "config": dict(cfg), "targets": TARGETS, "vocabs": VOCABS,
                "bpp_feature_names": BPP_FEATURE_NAMES if cfg.get("use_bpp", False) else [],
                "epoch": epoch, "val_metrics": val_metrics,
                "train_ids": list(train_loader.dataset.items[i]["id"] for i in range(len(train_loader.dataset))),
                "val_ids": list(val_loader.dataset.items[i]["id"] for i in range(len(val_loader.dataset))),
            }, best_path)
        else:
            stale += 1
        print(f"Epoch {epoch:02d} | train MSE {train_metrics['mse']:.4f} | "
              f"val MCRMSE {score:.4f} | degradation {val_metrics['degradation_mcrmse']:.4f} | "
              f"{row['seconds']:.1f}s" + (" | saved best" if improved else ""))
        if stale >= cfg["patience"]:
            print("Early stopping.")
            break
    checkpoint = torch.load(best_path, map_location="cpu", weights_only=True)
    model.load_state_dict(checkpoint["model_state"])
    model.to(device).eval()
    return pd.DataFrame(history), checkpoint

# Reset initialization and minibatch order if this cell is rerun.
torch.manual_seed(CFG["seed"])
generator.manual_seed(CFG["seed"])
model = ContinuousRNA(CFG).to(device)
history, best_checkpoint = fit(model, train_loader, val_loader, CFG, device, RUN_DIR)
print("Restored epoch", best_checkpoint["epoch"], "from", RUN_DIR / "best_model.pt")


## Review validation performance

These are internal grouped-validation scores, not competition leaderboard scores.
The validation set is used for model selection; run the final test section only after architecture choices are finished. The model below is the restored best checkpoint.


In [ ]:
best_metrics = run_epoch(model, val_loader, device)
comparison = pd.DataFrame({"train_mean_baseline": mean_baseline, "continuous_model": best_metrics})
print(comparison.round(5).to_string())
comparison.to_csv(RUN_DIR / "validation_metrics.csv")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(history.epoch, history.train_mse, label="Training")
axes[0].plot(history.epoch, history.val_mse, label="Validation")
axes[0].set(xlabel="Epoch", ylabel="Masked MSE", title="Training and validation loss")
axes[0].legend()
for target in TARGETS:
    axes[1].plot(history.epoch, history[f"val_rmse_{target}"], label=target)
axes[1].set(xlabel="Epoch", ylabel="RMSE", title="Validation error per target")
axes[1].legend(fontsize=8)
fig.tight_layout()
fig.savefig(RUN_DIR / "learning_curves.png", dpi=150)
plt.show()


## Export predictions and continuous embeddings

Save only real nucleotides, with one row per position. Unobserved ground truth is
stored as NaN. Embedding rows match the CSV rows exactly; `rna_id` and `position`
are also saved in the NPZ to make later FSQ analysis unambiguous. All 107 positions
have predictions/embeddings, including the unmeasured tail.


In [ ]:
@torch.no_grad()
def export_validation(model, loader, device, run_dir):
    model.eval()
    tables, embedding_parts = [], []
    for cpu_batch in loader:
        batch = move_batch(cpu_batch, device)
        embeddings = model.encode(batch["tokens"], batch["pairs"], batch["valid"],
                         batch["bpp"], batch["bpp_features"])
        pred = model.head(model.bottleneck(embeddings)).cpu().numpy()
        embeddings = embeddings.cpu().numpy()
        for i, rna_id in enumerate(cpu_batch["ids"]):
            n = int(cpu_batch["valid"][i].sum())
            part = pd.DataFrame({"id": [rna_id] * n, "position": np.arange(n)})
            labels = cpu_batch["labels"][i, :n].numpy()
            mask = cpu_batch["observed"][i, :n].numpy()
            for k, target in enumerate(TARGETS):
                part[f"true_{target}"] = np.where(mask[:, k], labels[:, k], np.nan)
                part[f"pred_{target}"] = pred[i, :n, k]
            tables.append(part)
            embedding_parts.append(embeddings[i, :n])
    predictions = pd.concat(tables, ignore_index=True)
    predictions.to_csv(run_dir / "validation_predictions.csv", index=False)
    np.savez_compressed(run_dir / "validation_embeddings.npz",
                        embeddings=np.concatenate(embedding_parts),
                        rna_id=predictions.id.to_numpy(dtype=str),
                        position=predictions.position.to_numpy())
    return predictions

predictions = export_validation(model, val_loader, device, RUN_DIR)
example_id = val_df.id.iloc[0]  # fixed first validation RNA, not cherry-picked
example = predictions.loc[predictions.id.eq(example_id)]
fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
for ax, target in zip(axes, TARGETS):
    ax.plot(example.position, example[f"true_{target}"], label="Measured", linewidth=1.5)
    ax.plot(example.position, example[f"pred_{target}"], label="Predicted", linewidth=1.2)
    ax.set(ylabel="Measurement", title=target)
    ax.legend()
axes[-1].set_xlabel("Nucleotide position (0-based)")
fig.suptitle(f"Validation RNA: {example_id}")
fig.tight_layout()
fig.savefig(RUN_DIR / "example_predictions.png", dpi=150)
plt.show()
print("Saved checkpoint, configuration, split, history, scores, predictions and embeddings to:", RUN_DIR)


## Evaluate the best checkpoint on public and private test sets

Run this section after selecting the model using validation only. It reloads
`best_model.pt` from disk, so scores cannot accidentally use the last training epoch
or an altered in-memory model. To evaluate a previous run, change `TEST_CHECKPOINT`.
Run the setup, dataset, model and metric definition cells first; retraining is not required.

`test_with_labels.json` contains 107-nt public and 130-nt private test RNAs.
Report both sets separately, using each row's `seq_scored` and finite-label mask.
The file has no `SN_filter`, so these scores use **all labeled RNAs in each set**
and are not claimed to reproduce the competition's filtered leaderboard scores.
No test results are used for checkpoint selection or learning-rate updates.


In [ ]:
TEST_CHECKPOINT = RUN_DIR / "best_model.pt"
# For a previous run, use its actual path instead:
# TEST_CHECKPOINT = Path("/absolute/path/to/previous/run/best_model.pt")

test_path = DATA / "test_with_labels.json"
if not test_path.exists():
    raise FileNotFoundError(f"Labeled test data not found: {test_path}")
if not TEST_CHECKPOINT.exists():
    raise FileNotFoundError(f"Best checkpoint not found: {TEST_CHECKPOINT}")

test_checkpoint = torch.load(TEST_CHECKPOINT, map_location="cpu", weights_only=True)
assert test_checkpoint["targets"] == TARGETS, "Checkpoint target order differs"
assert test_checkpoint["vocabs"] == VOCABS, "Checkpoint token vocabularies differ"
if test_checkpoint["config"].get("use_bpp", False):
    assert test_checkpoint.get("bpp_feature_names") == BPP_FEATURE_NAMES, "BPP feature schema differs"
test_model = ContinuousRNA(test_checkpoint["config"])
test_model.load_state_dict(test_checkpoint["model_state"])
test_model.to(device).eval()

test_df = pd.read_json(test_path, lines=True)
assert test_df.id.is_unique, "Duplicate test IDs"
assert set(test_df.seq_length) == {107, 130}, "Unexpected test sequence lengths"
seen_ids = set(test_checkpoint["train_ids"]) | set(test_checkpoint["val_ids"])
assert set(test_df.id).isdisjoint(seen_ids), "Test IDs overlap checkpoint training/validation IDs"

test_results = []
for name, length in (("public", 107), ("private", 130)):
    subset = test_df.loc[test_df.seq_length.eq(length)].reset_index(drop=True)
    dataset = RNADataset(subset, DATA / "bpps",
                         use_bpp=test_checkpoint["config"].get("use_bpp", False))
    loader = DataLoader(dataset, batch_size=test_checkpoint["config"]["batch_size"],
                        shuffle=False, collate_fn=collate_rna, num_workers=0)
    metrics = run_epoch(test_model, loader, device)
    result = {"test_set": name, "n_rnas": len(dataset), **metrics}
    for k, target in enumerate(TARGETS):
        result[f"n_observed_{target}"] = sum(
            int(item["observed"][:, k].sum()) for item in dataset.items
        )
    test_results.append(result)

test_results = pd.DataFrame(test_results).set_index("test_set")
print(f"Best checkpoint: {TEST_CHECKPOINT}")
print(f"Selected epoch: {test_checkpoint['epoch']} | "
      f"validation MCRMSE: {test_checkpoint['val_metrics']['mcrmse']:.5f}")
print("Test evaluation: all labeled RNAs; no quality filter available")
report_columns = ["n_rnas", "mcrmse", "degradation_mcrmse"] + [f"rmse_{t}" for t in TARGETS]
print(test_results[report_columns].round(5).to_string())
# Save beside the checkpoint, including the denominator for each target.
test_metrics_path = TEST_CHECKPOINT.parent / "test_metrics.csv"
test_results.to_csv(test_metrics_path)
print("Saved test metrics:", test_metrics_path)


## Private-test RMSE against DegScore-XGBoost

Score the selected checkpoint and the saved DegScore-XGBoost predictions on the private test.
`all` is every private RNA. `test_filter == 1` is the subset marked for evaluation in the local release.
RMSE uses each RNA's scored prefix and finite labels. The XGBoost checkpoint emits one number per nucleotide;
each of its columns compares that same number with a different measurement.

In [1]:
private_path = DATA.parent / "private_test" / "private_test.json"
xgb_path = (PROJECT / "RNADeg" / "baselines" / "degscore_xgboost"
            / "results" / "predictions" / "private_test_xgb.pt")
if not private_path.exists():
    raise FileNotFoundError(private_path)
if not xgb_path.exists():
    raise FileNotFoundError(xgb_path)

checkpoint_path = RUN_DIR / "best_model.pt"
if not checkpoint_path.exists():
    saved = sorted((PROJECT / "outputs" / "continuous_rna").glob("*/best_model.pt"))
    if not saved:
        raise FileNotFoundError("No saved continuous-model checkpoint")
    checkpoint_path = saved[-1]

checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
assert checkpoint["targets"] == TARGETS
assert checkpoint["vocabs"] == VOCABS
if checkpoint["config"].get("use_bpp", False):
    assert checkpoint.get("bpp_feature_names") == BPP_FEATURE_NAMES
private_model = ContinuousRNA(checkpoint["config"])
private_model.load_state_dict(checkpoint["model_state"])
private_model.to(device).eval()

private_df = pd.read_json(private_path, lines=True).reset_index(drop=True)
assert private_df.id.is_unique and private_df.seq_length.eq(130).all()
assert set(private_df.test_filter) <= {0, 1}
xgb_pred = torch.load(xgb_path, map_location="cpu", weights_only=True).numpy()
assert xgb_pred.shape[:2] == (len(private_df), int(private_df.seq_length.max()))
assert xgb_pred.shape[-1] == 1

@torch.no_grad()
def predict_private(model, frame):
    dataset = RNADataset(frame, BPP_DIR, use_bpp=model.use_bpp)
    loader = DataLoader(dataset, batch_size=checkpoint["config"]["batch_size"],
                        shuffle=False, collate_fn=collate_rna, num_workers=0)
    chunks = []
    for cpu_batch in loader:
        batch = move_batch(cpu_batch, device)
        pred = model(batch["tokens"], batch["pairs"], batch["valid"],
                     batch["bpp"], batch["bpp_features"])
        chunks.append(pred.cpu().numpy())
    return np.concatenate(chunks, axis=0)

continuous_pred = predict_private(private_model, private_df)
scored = int(private_df.seq_scored.iloc[0])
assert private_df.seq_scored.eq(scored).all()
labels = np.stack([
    np.asarray([row[:scored] for row in private_df[target]], dtype=np.float64)
    for target in TARGETS
], axis=-1)
finite = np.isfinite(labels)

def rmse_table(predictions, mask):
    error = np.square(predictions[mask, :scored] - labels[mask])
    error = np.where(finite[mask], error, 0.0)
    counts = finite[mask].sum(axis=(0, 1))
    if (counts == 0).any():
        raise ValueError("A target has no observed private-test labels in this subset")
    rmse = np.sqrt(error.sum(axis=(0, 1)) / counts)
    row = {f"rmse_{target}": float(rmse[k]) for k, target in enumerate(TARGETS)}
    row["mcrmse"] = float(rmse.mean())
    row["n_rnas"] = int(mask.sum())
    row["n_positions"] = int(counts[0])
    return row

xgb_as_targets = np.repeat(xgb_pred[:, :, :1], len(TARGETS), axis=-1)
subsets = {
    "all": np.ones(len(private_df), dtype=bool),
    "test_filter == 1": private_df.test_filter.eq(1).to_numpy(),
}
rows = []
for subset, mask in subsets.items():
    for model_name, pred in (("continuous", continuous_pred), ("xgboost", xgb_as_targets)):
        rows.append({"subset": subset, "model": model_name, **rmse_table(pred, mask)})

comparison = pd.DataFrame(rows).set_index(["subset", "model"])
columns = ["n_rnas", "n_positions", *[f"rmse_{target}" for target in TARGETS], "mcrmse"]
print(f"Continuous checkpoint: {checkpoint_path}")
print(f"XGBoost predictions: {xgb_path}")
print("Private-test RMSE. XGBoost has one output, compared separately with each target.")
print(comparison[columns].round(5).to_string())
comparison_path = checkpoint_path.parent / "private_test_comparison.csv"
comparison.to_csv(comparison_path)
print("Saved:", comparison_path)


Continuous checkpoint: /Users/siddharth/MS/ML4FG/Project/outputs/continuous_rna/20261111_233855_690768/best_model.pt
XGBoost predictions: /Users/siddharth/MS/ML4FG/Project/RNADeg/baselines/degscore_xgboost/results/predictions/private_test_xgb.pt
Private-test RMSE. XGBoost has one output, compared separately with each target.
                             n_rnas  n_positions  rmse_reactivity  rmse_deg_Mg_pH10  rmse_deg_Mg_50C   mcrmse
subset           model                                                                                       
all              continuous    3005       273455          0.36944           0.57864          0.43877  0.46229
                 xgboost       3005       273455          0.43186           0.57082          0.45091  0.48453
test_filter == 1 continuous    1172       106652          0.30152           0.54845          0.39004  0.41333
                 xgboost       1172       106652          0.39605           0.56243          0.42127  0.45992
Saved: /Users

## Next controlled experiments

1. Keep this split and compare against the existing DegScore-XGBoost baseline.
2. Add a small bounded continuous bottleneck before the prediction head.
3. Quantize that same bottleneck with FSQ, holding the encoder, head, targets,
   training budget and selection criterion fixed. Use multiple seeds for conclusions.
4. Add Ribonanza reactivity pretraining with a separate DMS/2A3 head, then fine-tune
   on these degradation targets. Check sequence overlap before pretraining.

Prediction quality and interpretability are separate questions: examine code associations
within nucleotide and loop categories on held-out RNAs, not just training examples.

For a BPP ablation, set `CFG["use_bpp"] = False` and rerun all cells with the same seed and split. The original model architecture is retained in that mode, and pre-BPP checkpoints remain loadable.
